In [1]:
from openai import OpenAI

BASE_URL = "http://localhost:11434/v1"
MODELL = "llama3.2"

client = OpenAI(base_url=BASE_URL, api_key="ollama")

In [2]:
antwort = client.chat.completions.create(
    model=MODELL,
    messages=[
        {"role": "system", "content": "Du bist der Einkaufsassistent des Hotel Seeblick. Antworte kurz auf Deutsch."},
        {"role": "user", "content": "Was ist ein Mindestbestand?"},
    ],
)
print(antwort.choices[0].message.content)

Ein Mindestbestand ist ein bestimmtes Maß an Produkten oder Dienstleistungen, die ein Geschäft oder Unternehmen aufrechterhalten muss, um seine Verpflichtungen zu erfüllen und seine Kunden zu befriedigen. Im Zusammenhang mit dem Hotel Seeblick könnte dies beispielsweise die Anzahl der verfügbaren Zimmer, Gerichte in der Küche oder Dienstleistungen wie Reiserouten anbieten.


In [7]:
from typing import Literal

from pydantic import BaseModel, Field


class Bedarf(BaseModel):
    artikel: str = Field(description="Name des Artikels, z. B. Mehl")
    menge: int = Field(gt=0, description="Bestellmenge als ganze Zahl")
    einheit: Literal["kg", "Liter", "Stück", "Packung"] = Field(
        description="Einheit; bei zählbaren Dingen wie Eiern immer Stück"
    )


class Bestellanfrage(BaseModel):
    abteilung: Literal["Küche", "Housekeeping", "Rezeption"] = Field(
        description="Die Abteilung, die die Ware braucht – nicht der Empfänger der Nachricht"
    )
    dringend: bool
    bedarfe: list[Bedarf]

In [11]:
nachricht = """Hallo Einkauf,
die Küche braucht bis Freitag dringend 30 kg Mehl, 200 Eier und 5 Liter Olivenöl.
Grüße, Chefkoch"""

antwort = client.chat.completions.parse(
    model=MODELL,
    messages=[
        {"role": "system", "content": "Du bist der Einkaufsassistent des Hotel Seeblick." 
            "Extrahiere aus der Nachricht eine Bestellanfrage. Die Abteilung ist der Absender, nicht der Einkauf."},
        {"role": "user", "content": nachricht},
    ],
    response_format=Bestellanfrage,
)

anfrage = antwort.choices[0].message.parsed
print(anfrage)
print()
for bedarf in anfrage.bedarfe:
    print(f"{bedarf.menge} {bedarf.einheit} {bedarf.artikel}")

abteilung='Küche' dringend=True bedarfe=[Bedarf(artikel='Mehl', menge=30, einheit='kg'), Bedarf(artikel='Eier', menge=200, einheit='Stück'), Bedarf(artikel='Olivenöl', menge=5, einheit='Liter')]

30 kg Mehl
200 Stück Eier
5 Liter Olivenöl
